# KQA Pro × Grounded SELL-Horn Experiments (Executable)
Real **KQA Pro** questions + gold **KoPL programs** compiled into a grounded, resource-aware proof-search instance.

Evaluates:
1. **No unauthorized reuse of observations** (linearity)
2. **Budget-correctness** (bounded firings)
3. **Proof-carrying verification** (independent trace replay)

Outputs written to `sell_experiments_out/`:
- `table_main.tex`
- `table_ablations.tex`
- `stress_grid.csv`


In [1]:
# 0) Setup (lightweight deps)
import sys, subprocess, importlib.util
def ensure(pkg: str):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure("datasets")
ensure("tqdm")

from datasets import load_dataset
from tqdm import tqdm

from dataclasses import dataclass
from typing import Dict, List, Tuple, Iterable, Optional, Set, FrozenSet
from collections import Counter, defaultdict
import re, json, random, time, csv
from pathlib import Path

SEED = 7
random.seed(SEED)

# Use home directory for writable output (GitHub VFS is read-only)
OUT_DIR = Path.home() / "sell_experiments_out"
OUT_DIR.mkdir(exist_ok=True)

def now_ms() -> int:
    return int(time.time() * 1000)

print("Output dir:", OUT_DIR.resolve())

Output dir: /Users/cero/sell_experiments_out


## 1) Load KQA Pro (train/val)
We use the HuggingFace mirror `drt/kqa_pro` with fields:
`question`, `program` (KoPL), `answer`, `choices`, `sparql` (train/val).

In [ ]:
# 1) Load dataset using huggingface_hub to download raw files
# The HuggingFace datasets loader script is no longer supported.
# We download the raw data files directly from the HF repo.

from huggingface_hub import hf_hub_download
import os

KQA_LOCAL = Path.home() / "kqa_pro_data"
KQA_LOCAL.mkdir(exist_ok=True)

def download_kqa_pro():
    train_path = KQA_LOCAL / "train.json"
    val_path = KQA_LOCAL / "val.json"
    
    if not train_path.exists():
        print("Downloading train.json from HuggingFace...")
        downloaded = hf_hub_download(
            repo_id="drt/kqa_pro",
            filename="train.json",
            repo_type="dataset",
            local_dir=KQA_LOCAL
        )
        print(f"Downloaded to {downloaded}")
    
    if not val_path.exists():
        print("Downloading val.json from HuggingFace...")
        downloaded = hf_hub_download(
            repo_id="drt/kqa_pro",
            filename="val.json",
            repo_type="dataset",
            local_dir=KQA_LOCAL
        )
        print(f"Downloaded to {downloaded}")
    
    return KQA_LOCAL

kqa_path = download_kqa_pro()

# Load train and val JSON files
with open(kqa_path / "train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(kqa_path / "val.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)

# Convert to dataset-like objects for compatibility
from datasets import Dataset

train = Dataset.from_list(train_data)
val = Dataset.from_list(val_data)

print("Train size:", len(train))
print("Val size:", len(val))
print("Fields:", train.column_names)

# Default experiment subset (increase for paper runs)
N_VAL = 2000
val_subset = val.select(range(min(N_VAL, len(val))))
print("Val subset:", len(val_subset))

HTTPError: HTTP Error 404: Not Found

## 2) Compile KoPL into grounded budgeted rules
Each KoPL step i becomes a grounded rule firing that consumes:
- exactly one **budget token** `B`
- exactly one **evidence token** `ev:<fn>:...` (linear observation)
- intermediate dependencies `v<dep>` (linear, derived)

The final rule emits `ans:<gold_answer>`.
This lets you validate SELL’s **resource discipline** on a real KBQA benchmark with explicit programs.

In [ ]:
Atom = str
BUDGET_ATOM: Atom = "B"

@dataclass(frozen=True)
class Rule:
    rid: str
    head: Atom
    body_persistent: Tuple[Atom, ...]
    body_linear: Tuple[Atom, ...]

@dataclass
class ProofResult:
    proved: bool
    answer: Optional[Atom]
    trace: List[str]
    time_ms: int
    debug: Dict

def tok(s: str) -> List[str]:
    return re.findall(r"[A-Za-z0-9_:\\-]+", str(s))

def evidence_token(step_fn: str, inputs: List[str]) -> Atom:
    parts = [step_fn] + [t for x in inputs for t in tok(x)[:2]]
    return "ev:" + ":".join(parts[:4])

def compile_program_to_rules(program: List[Dict], answer_str: str) -> Tuple[List[Rule], Atom, Set[Atom], List[Atom]]:
    rules: List[Rule] = []
    persistent: Set[Atom] = set()
    evidence_vocab: List[Atom] = []
    for i, step in enumerate(program):
        fn = step["function"]
        deps = step.get("dependencies", [])
        inp = step.get("inputs", [])
        ev = evidence_token(fn, inp)
        evidence_vocab.append(ev)

        head = f"v{i}"
        dep_atoms = tuple(f"v{d}" for d in deps)
        body_linear = (ev,) + dep_atoms

        # optional persistent constants for debugging
        for x in inp:
            persistent.add("c:" + ":".join(tok(x)[:4]))

        rules.append(Rule(rid=f"r{i}:{fn}", head=head, body_persistent=tuple(), body_linear=body_linear))

    goal = f"ans:{answer_str}"
    if program:
        rules.append(Rule(rid="r_last:emit", head=goal, body_persistent=tuple(), body_linear=(f"v{len(program)-1}",)))
    else:
        persistent.add(goal)

    evidence_vocab = sorted(set(evidence_vocab))
    return rules, goal, persistent, evidence_vocab

ex0 = val_subset[0]
rules0, goal0, pers0, vocab0 = compile_program_to_rules(ex0["program"], ex0["answer"])
print("Q:", ex0["question"])
print("Goal:", goal0)
print("Program steps:", len(ex0["program"]))
print("Evidence vocab size:", len(vocab0))
print("First rule:", rules0[0])


## 3) Grounded SELL-Horn prover + independent certificate replay

In [ ]:
@dataclass(frozen=True)
class ProverConfig:
    max_depth: int = 256
    beam_width: int = 400
    use_caching: bool = True

def multiset_contains(ms: Counter, required: Iterable[Atom]) -> bool:
    req = Counter(required)
    return all(ms[a] >= c for a, c in req.items())

def multiset_remove(ms: Counter, items: Iterable[Atom]) -> Counter:
    ms2 = ms.copy()
    for a in items:
        ms2[a] -= 1
        if ms2[a] <= 0:
            del ms2[a]
    return ms2

def counter_frozenset(ms: Counter) -> FrozenSet[Tuple[Atom, int]]:
    return frozenset(ms.items())

def prove(goal: Atom, persistent: Set[Atom], S: Counter, rules_by_head: Dict[Atom, List[Rule]],
          cfg: ProverConfig, depth: int, trace: List[str],
          cache: Optional[Set[Tuple[Atom, FrozenSet[Tuple[Atom,int]], int]]] = None) -> Optional[Tuple[List[str], Counter]]:
    if depth > cfg.max_depth:
        return None
    if goal in persistent:
        return (trace, S)

    # linear axiom
    if S.get(goal, 0) > 0:
        S2 = S.copy()
        S2[goal] -= 1
        if S2[goal] == 0: del S2[goal]
        return (trace, S2)

    rem_budget = S.get(BUDGET_ATOM, 0)
    if cfg.use_caching and cache is not None:
        key = (goal, counter_frozenset(S), rem_budget)
        if key in cache:
            return None
        cache.add(key)

    candidates = sorted(rules_by_head.get(goal, []), key=lambda r: (len(r.body_linear), r.rid))
    for r in candidates[:cfg.beam_width]:
        if S.get(BUDGET_ATOM, 0) <= 0:
            continue
        if not all(a in persistent for a in r.body_persistent):
            continue
        if not multiset_contains(S, r.body_linear):
            continue

        S_after = multiset_remove(S, (BUDGET_ATOM,))
        S_after = multiset_remove(S_after, r.body_linear)
        S_after.update([r.head])

        out = prove(goal, persistent, S_after, rules_by_head, cfg, depth+1, trace+[r.rid], cache)
        if out is not None:
            return out
    return None

def run_prover(goal: Atom, persistent: Set[Atom], observations: List[Atom], k_budget: int,
              rules: List[Rule], cfg: ProverConfig) -> ProofResult:
    t0 = int(time.time()*1000)
    S = Counter(observations)
    S.update([BUDGET_ATOM]*k_budget)

    rules_by_head = defaultdict(list)
    for r in rules:
        rules_by_head[r.head].append(r)

    cache = set() if cfg.use_caching else None
    res = prove(goal, persistent, S, rules_by_head, cfg, 0, [], cache)
    t1 = int(time.time()*1000)

    if res is None:
        return ProofResult(False, None, [], t1-t0, {"cache_size": len(cache) if cache is not None else None})

    trace, S_final = res
    ok = (goal in persistent) or (S_final.get(goal,0)>0)
    return ProofResult(ok, goal if ok else None, trace, t1-t0, {"cache_size": len(cache) if cache is not None else None})

def verify_trace(goal: Atom, persistent: Set[Atom], observations: List[Atom], k_budget: int,
                 rules_by_id: Dict[str, Rule], trace: List[str]) -> Dict:
    t0 = int(time.time()*1000)
    S = Counter(observations)
    S.update([BUDGET_ATOM]*k_budget)

    obs_origin = Counter(observations)
    budget_used = 0
    obs_consumed = 0

    for rid in trace:
        if rid not in rules_by_id:
            return {"ok": False, "reason": f"unknown rule {rid}"}
        r = rules_by_id[rid]

        if S.get(BUDGET_ATOM,0) <= 0:
            return {"ok": False, "reason": "budget underflow", "at": rid}
        if not all(a in persistent for a in r.body_persistent):
            return {"ok": False, "reason": "missing persistent premise", "at": rid}
        if not multiset_contains(S, r.body_linear):
            return {"ok": False, "reason": "linear premise underflow (reuse?)", "at": rid}

        S = multiset_remove(S, (BUDGET_ATOM,))
        S = multiset_remove(S, r.body_linear)
        budget_used += 1

        for a in r.body_linear:
            if a.startswith("ev:") and obs_origin[a] > 0:
                obs_origin[a] -= 1
                obs_consumed += 1

        S.update([r.head])

    ok = (goal in persistent) or (S.get(goal,0) > 0)
    t1 = int(time.time()*1000)
    return {
        "ok": ok,
        "trace_len": len(trace),
        "budget_used": budget_used,
        "obs_consumed": obs_consumed,
        "verification_time_ms": t1-t0
    }

def check_budget_correctness(trace: List[str], k: int) -> Dict:
    return {"pass": len(trace) <= k, "k": k, "trace_len": len(trace)}

def check_no_observation_reuse(trace: List[str], observations: List[Atom], rules_by_id: Dict[str, Rule]) -> Dict:
    S = Counter(observations)
    for rid in trace:
        r = rules_by_id[rid]
        evs = [a for a in r.body_linear if a.startswith("ev:")]
        if not multiset_contains(S, evs):
            return {"pass": False, "reason": "evidence token reuse/underflow", "at": rid}
        S = multiset_remove(S, evs)
    return {"pass": True}


## 4) Evidence proposer (top-K + noise η)
We simulate the neural component as a proposer of evidence tokens; noise η swaps out a fraction with distractors.

In [ ]:
def tokenize(text: str) -> Set[str]:
    return set(re.findall(r"[A-Za-z0-9_]+", text.lower()))

def proposer_topk(question: str, evidence_vocab: List[Atom], K: int, eta: float, rng: random.Random) -> List[Atom]:
    qtok = tokenize(question)
    def score(ev: Atom) -> int:
        return len(qtok & tokenize(ev))
    ranked = sorted(evidence_vocab, key=lambda e: (score(e), rng.random()), reverse=True)
    top = ranked[:K]
    if eta > 0:
        n = max(0, int(round(eta * len(top))))
        distractors = [e for e in evidence_vocab if e not in top]
        rng.shuffle(distractors)
        for i in range(min(n, len(top), len(distractors))):
            top[-(i+1)] = distractors[i]
    return top


## 5) Systems and ablations

In [ ]:
@dataclass(frozen=True)
class ExperimentSetting:
    name: str
    obs_consumable: bool
    use_budget: bool
    obs_persistent: bool = False

cfg = ProverConfig()

def run_instance(question: str, program: List[Dict], answer: str,
                 setting: ExperimentSetting, K: int, k_budget: int, eta: float,
                 cfg: ProverConfig, rng: random.Random) -> Tuple[ProofResult, Dict]:
    rules, goal, persistent, ev_vocab = compile_program_to_rules(program, answer)
    rules_by_id = {r.rid: r for r in rules}

    obs = proposer_topk(question, ev_vocab, K=K, eta=eta, rng=rng)

    persistent_aug = set(persistent)
    obs_linear: List[Atom] = []

    if setting.obs_persistent:
        persistent_aug.update(obs)
    else:
        if setting.obs_consumable:
            obs_linear = obs
        else:
            persistent_aug.update(obs)

    k_eff = k_budget if setting.use_budget else 10**9

    pr = run_prover(goal, persistent_aug, obs_linear, k_eff, rules, cfg)
    cert = verify_trace(goal, persistent_aug, obs_linear, k_eff, rules_by_id, pr.trace) if pr.proved else {"ok": False}
    checks = {}
    if pr.proved:
        checks["budget"] = check_budget_correctness(pr.trace, k_eff)
        checks["no_obs_reuse"] = check_no_observation_reuse(pr.trace, obs_linear, rules_by_id)

    return pr, {"goal": goal, "obs": obs, "cert": cert, "checks": checks, "n_steps": len(program), "n_vocab": len(ev_vocab)}


## 6) Metrics + LaTeX export

In [ ]:
def exact_match(pred: Optional[Atom], gold: str) -> float:
    return 1.0 if pred == f"ans:{gold}" else 0.0

def aggregate(rows):
    n = len(rows)
    em = proof = ver = 0.0
    tms = bud = obs = 0.0
    reuse_viol = 0
    for gold, pr, aux in rows:
        if pr.proved:
            proof += 1.0
            em += exact_match(pr.answer, gold)
            ver += 1.0 if aux["cert"].get("ok", False) else 0.0
            bud += aux["cert"].get("budget_used", 0)
            obs += aux["cert"].get("obs_consumed", 0)
            if not aux["checks"]["no_obs_reuse"]["pass"]:
                reuse_viol += 1
        tms += pr.time_ms
    return {"n": n, "EM": em/n if n else 0.0, "ProofRate": proof/n if n else 0.0, "VerifRate": ver/n if n else 0.0,
            "AvgTimeMs": tms/n if n else 0.0, "AvgBud": bud/n if n else 0.0, "AvgObs": obs/n if n else 0.0,
            "ObsReuseViolations": reuse_viol}

def latex_table_main(rows):
    lines = [r"\begin{tabular}{lrrrrrrr}", r"\toprule",
             r"System & EM & Proof\_rate & Verif\_rate & AvgTime(ms) & AvgBud & AvgObs & ObsReuseV\\",
             r"\midrule"]
    for name, m in rows:
        lines.append(f"{name} & {m['EM']:.2f} & {m['ProofRate']:.2f} & {m['VerifRate']:.2f} & {m['AvgTimeMs']:.1f} & {m['AvgBud']:.2f} & {m['AvgObs']:.2f} & {m['ObsReuseViolations']}\\\\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(lines)

def latex_table_abl(rows):
    lines = [r"\begin{tabular}{lrrrrrr}", r"\toprule",
             r"Ablation & EM & Proof\_rate & AvgTime(ms) & AvgBud & AvgObs & ObsReuseV\\",
             r"\midrule"]
    for name, m in rows:
        lines.append(f"{name} & {m['EM']:.2f} & {m['ProofRate']:.2f} & {m['AvgTimeMs']:.1f} & {m['AvgBud']:.2f} & {m['AvgObs']:.2f} & {m['ObsReuseViolations']}\\\\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    return "\n".join(lines)


## 7) Main experiment (KQA Pro val subset)

In [ ]:
def run_eval(split, setting: ExperimentSetting, K: int, k_cap: int, eta: float, seed: int, limit: int):
    rng = random.Random(SEED + seed)
    out = []
    for i in tqdm(range(min(limit, len(split))), desc=setting.name):
        ex = split[i]
        prog = ex["program"]
        ans = ex["answer"]
        k_budget = min(len(prog) + 1, k_cap)
        pr, aux = run_instance(ex["question"], prog, ans, setting, K=K, k_budget=k_budget, eta=eta, cfg=cfg, rng=rng)
        if pr.proved and setting.name == "SELL":
            assert aux["checks"]["budget"]["pass"], aux["checks"]["budget"]
            assert aux["checks"]["no_obs_reuse"]["pass"], aux["checks"]["no_obs_reuse"]
        out.append((ans, pr, aux))
    return out

LIMIT = 500
K = 25
ETA = 0.2
K_CAP = 64

main_systems = [
    ("Neural-only baseline", None),
    ("NeSy baseline (evidence persistent)", ExperimentSetting("NeSy", obs_consumable=False, use_budget=False)),
    ("Ours (SELL)", ExperimentSetting("SELL", obs_consumable=True, use_budget=True)),
]

def run_neural_only(split, limit):
    return [(split[i]["answer"], ProofResult(False, None, [], 0, {}), {"cert":{"ok":False}, "checks":{}})
            for i in range(min(limit, len(split)))]

rows = []
res_neural = run_neural_only(val_subset, LIMIT)
rows.append(("Neural-only baseline", aggregate(res_neural)))

res_nesy = run_eval(val_subset, main_systems[1][1], K=K, k_cap=K_CAP, eta=ETA, seed=0, limit=LIMIT)
rows.append((main_systems[1][0], aggregate(res_nesy)))

res_sell = run_eval(val_subset, main_systems[2][1], K=K, k_cap=K_CAP, eta=ETA, seed=1, limit=LIMIT)
rows.append((main_systems[2][0], aggregate(res_sell)))

tex_main = latex_table_main(rows)
print(tex_main)
(OUT_DIR/"table_main.tex").write_text(tex_main, encoding="utf-8")


## 8) Ablations

In [ ]:
ablations = [
    ("Ours (SELL)", ExperimentSetting("SELL", True, True)),
    ("No budget", ExperimentSetting("NoBudget", True, False)),
    ("All persistent (no subexp discipline)", ExperimentSetting("AllPersistent", False, False)),
    ("Obs persistent (allows reuse)", ExperimentSetting("ObsPersistent", True, True, obs_persistent=True)),
]

abl_rows = []
for name, setting in ablations:
    res = run_eval(val_subset, setting, K=K, k_cap=K_CAP, eta=ETA, seed=10, limit=LIMIT)
    abl_rows.append((name, aggregate(res)))

tex_abl = latex_table_abl(abl_rows)
print(tex_abl)
(OUT_DIR/"table_ablations.tex").write_text(tex_abl, encoding="utf-8")


## 9) Stress grid → CSV

In [ ]:
def stress_grid(Ks=(10,25,50), k_caps=(16,32,64), etas=(0.0,0.2,0.4), limit=200):
    setting = ExperimentSetting("SELL", True, True)
    rows = []
    for K_ in Ks:
        for kc in k_caps:
            for eta_ in etas:
                res = run_eval(val_subset, setting, K=K_, k_cap=kc, eta=eta_, seed=100 + int(1000*eta_) + K_ + kc, limit=limit)
                m = aggregate(res)
                rows.append({"K": K_, "k_cap": kc, "eta": eta_, **m})
    return rows

grid = stress_grid(limit=200)
csv_path = OUT_DIR/"stress_grid.csv"
with csv_path.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(grid[0].keys()))
    w.writeheader()
    w.writerows(grid)

print("Wrote:", csv_path.resolve())
print("First rows:", grid[:3])


## 10) Inspect one solved proof (trace + verification)

In [ ]:
setting = ExperimentSetting("SELL", True, True)
rng = random.Random(SEED + 999)

for idx in range(0, 50):
    ex = val_subset[idx]
    k_budget = min(len(ex["program"]) + 1, 64)
    pr, aux = run_instance(ex["question"], ex["program"], ex["answer"],
                           setting, K=50, k_budget=k_budget, eta=0.0, cfg=cfg, rng=rng)
    if pr.proved:
        print("IDX:", idx)
        print("Q:", ex["question"])
        print("Gold:", ex["answer"])
        print("Steps:", len(ex["program"]), "Budget:", k_budget)
        print("Trace len:", len(pr.trace))
        print("Verifier:", aux["cert"])
        print("Checks:", aux["checks"])
        print("Trace head:", pr.trace[:10])
        break
else:
    print("No proof found in first 50 with current K/eta. Try increasing K or lowering eta.")
